# Day 042 — Exercise 4: group_revenue

**What you'll build:** `group_revenue(conn, group_col) -> list[dict]` — aggregate orders by any column using `GROUP BY`, returning total, order count, and average revenue per group.

**Why it matters:** `GROUP BY` in SQL is what `df.groupby().agg()` is in pandas — but it runs inside the database, which is more efficient for large datasets. `SUM()`, `COUNT()`, `AVG()`, `MIN()`, `MAX()` are the standard SQL aggregate functions. `ORDER BY total DESC` sorts the results largest-first.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sqlite3
import pandas as pd


import sqlite3

def setup_db(conn):
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id  INTEGER PRIMARY KEY,
            product   TEXT,
            category  TEXT,
            region    TEXT,
            price     REAL,
            quantity  INTEGER,
            revenue   REAL
        )''')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product    TEXT PRIMARY KEY,
            category   TEXT,
            unit_price REAL
        )''')
    rows = [
        (1,'Widget','Electronics','North',25.0,10,250.0),
        (2,'Gadget','Electronics','South',150.0,3,450.0),
        (3,'Widget','Electronics','South',25.0,5,125.0),
        (4,'Doohickey','Accessories','East',8.0,50,400.0),
        (5,'Gadget','Electronics','East',150.0,7,1050.0),
        (6,'Widget','Electronics','East',25.0,4,100.0),
        (7,'Doohickey','Accessories','North',8.0,20,160.0),
        (8,'Gadget','Electronics','North',150.0,2,300.0),
        (9,'Widget','Electronics','West',25.0,6,150.0),
        (10,'Doohickey','Accessories','South',8.0,15,120.0),
        (11,'Thingamajig','Accessories','North',200.0,1,200.0),
        (12,'Thingamajig','Accessories','East',200.0,4,800.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', rows
    )
    products = [
        ('Widget','Electronics',25.0),
        ('Gadget','Electronics',150.0),
        ('Doohickey','Accessories',8.0),
        ('Thingamajig','Accessories',200.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO products VALUES (?,?,?)', products
    )
    conn.commit()


def run_query(conn, sql, params=()):
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [col[0] for col in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]


def filter_orders(conn, region=None, category=None, min_revenue=None):
    conditions = []
    params = []
    if region is not None:
        conditions.append('region = ?')
        params.append(region)
    if category is not None:
        conditions.append('category = ?')
        params.append(category)
    if min_revenue is not None:
        conditions.append('revenue >= ?')
        params.append(min_revenue)
    where = ('WHERE ' + ' AND '.join(conditions)) if conditions else ''
    sql = f'SELECT * FROM orders {where} ORDER BY order_id'
    return run_query(conn, sql, tuple(params))


conn = sqlite3.connect(':memory:')
setup_db(conn)

## Your Implementation

In [ ]:
def group_revenue(conn, group_col):
    """
    Aggregate orders by group_col, returning per-group revenue stats.

    SQL shape:
      SELECT {group_col},
             SUM(revenue)       AS total,
             COUNT(*)           AS orders,
             ROUND(AVG(revenue), 2) AS avg_revenue
      FROM orders
      GROUP BY {group_col}
      ORDER BY total DESC

    Returns:
        list[dict] — one dict per group, sorted by total desc
    """
    # TODO: build the SQL string using an f-string for group_col
    # TODO: return run_query(conn, sql)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'group_revenue' in globals()
        passed += 1; print('\u2705 Check 1: group_revenue is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a list of dicts
    try:
        result = group_revenue(conn, 'product')
        assert isinstance(result, list) and len(result) > 0
        assert isinstance(result[0], dict)
        passed += 1; print('\u2705 Check 2: returns list of dicts')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: group by product gives 4 groups, Gadget first
    try:
        assert len(result) == 4, f'expected 4 products, got {len(result)}'
        assert result[0]['product'] == 'Gadget', \
            f'Gadget should be first (highest total), got {result[0]["product"]}'
        assert abs(result[0]['total'] - 1800.0) < 0.01
        passed += 1; print('\u2705 Check 3: Gadget leads with 1800.0 total')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 4: result has expected columns
    try:
        expected_keys = {'product', 'total', 'orders', 'avg_revenue'}
        actual_keys = set(result[0].keys())
        assert actual_keys == expected_keys, \
            f'expected columns {expected_keys}, got {actual_keys}'
        passed += 1; print(f'\u2705 Check 4: correct columns {expected_keys}')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: group by region gives 4 groups
    try:
        by_region = group_revenue(conn, 'region')
        assert len(by_region) == 4, f'expected 4 regions, got {len(by_region)}'
        assert 'region' in by_region[0]
        passed += 1; print('\u2705 Check 5: group by region gives 4 groups')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def group_revenue(conn, group_col):
    sql = (
        f'SELECT {group_col}, SUM(revenue) AS total, '
        'COUNT(*) AS orders, ROUND(AVG(revenue), 2) AS avg_revenue '
        f'FROM orders GROUP BY {group_col} ORDER BY total DESC'
    )
    return run_query(conn, sql)
```

</details>